In [ ]:
import os
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# 1. FONCTION : explorer_arborescence
# ----------------------------------------------------------------------

def explorer_arborescence(path_folder_input: str, 
                         Niv: int = 3, 
                         with_files: bool = False) -> pd.DataFrame:
    """
    Explore une arborescence et génère un DataFrame représentant la hiérarchie.
    """
    if not os.path.isdir(path_folder_input):
        raise ValueError(f"Le dossier '{path_folder_input}' n'existe pas.")
    
    lignes = []

    def parcourir(repertoire, niveau_actuel, chemin_courant):
        if niveau_actuel > Niv:
            return
        
        try:
            elements = sorted(os.listdir(repertoire))
        except PermissionError:
            return
        
        padding_size_dossier = Niv - niveau_actuel
        
        for elem in elements:
            chemin_elem = os.path.join(repertoire, elem)
            
            if os.path.isdir(chemin_elem):
                # Ajout dossier : [chemin_parent] + [nom_dossier] + [padding vide]
                lignes.append(chemin_courant + [elem] + [""]*padding_size_dossier)
                # Récursion
                parcourir(chemin_elem, niveau_actuel+1, chemin_courant + [elem])
            elif with_files:
                # Ajout fichier : placé à la toute dernière colonne (Niveau_Niv)
                nb_colonnes_chemin = len(chemin_courant)
                padding_files = Niv - nb_colonnes_chemin - 1
                if padding_files >= 0:
                    lignes.append(chemin_courant + [""]*padding_files + [elem])
    
    parcourir(path_folder_input, 1, [])
    
    colonnes = [f"Niveau_{i}" for i in range(1, Niv+1)]
    df = pd.DataFrame(lignes, columns=colonnes) if lignes else pd.DataFrame(columns=colonnes)
    
    return df

# ----------------------------------------------------------------------
# 2. FONCTION GLOBALE : explorer_arborescences_global (CORRIGÉE)
# ----------------------------------------------------------------------

def explorer_arborescences_global(folders_to_list: list[list], 
                                  path_folder_output: str, 
                                  output_filename: str = "Arborescences_Globales.xlsx") -> None:
    """
    Explore plusieurs arborescences, gère les profondeurs mixtes et nettoie les résultats.
    """
    
    if not folders_to_list:
        print("La liste des dossiers à explorer est vide.")
        return

    # 1. Déterminer le niveau maximum global pour aligner les colonnes
    try:
        niv_max_global = max(item[1] for item in folders_to_list)
    except IndexError:
        print("Erreur : La configuration des dossiers est incorrecte.")
        return
        
    colonnes_finales = [f"Niveau_{i}" for i in range(1, niv_max_global + 1)]
    
    all_dfs = []

    # 2. Explorer chaque dossier et aligner les colonnes
    for config in folders_to_list:
        try:
            path_input, niv_local, with_files = config[0], config[1], config[2]
            
            # Exploration
            df_local = explorer_arborescence(path_input, niv_local, with_files)
            
            # Ajouter les colonnes manquantes si le dossier est moins profond que le max global
            for i in range(niv_local + 1, niv_max_global + 1):
                df_local[f"Niveau_{i}"] = "" 
            
            # Réordonner les colonnes
            df_local = df_local[colonnes_finales]
            all_dfs.append(df_local)

        except Exception as e:
            print(f"Erreur sur {config[0]} : {e}")
            continue

    if not all_dfs:
        print("Aucune arborescence n'a pu être explorée.")
        return

    # 3. Concaténer tout
    final_df = pd.concat(all_dfs, ignore_index=True)

    # 4. LOGIQUE DE NETTOYAGE CORRIGÉE
    # On conserve les lignes qui ont du contenu dans la dernière colonne (Niveau Max)
    # OU dans l'avant-dernière colonne (Niveau Max - 1).
    # Cela permet de garder à la fois les albums en profondeur 3 et ceux en profondeur 2.
    
    print("\nAjustement du DataFrame : Conservation des lignes avec contenu pertinent...")
    
    derniere_col = colonnes_finales[-1]
    avant_derniere_col = colonnes_finales[-2]
    
    # Filtre : Garde si Dernière Colonne est pleine OU Avant-Dernière est pleine
    mask = (final_df[derniere_col] != "") | (final_df[avant_derniere_col] != "")
    df_clean = final_df[mask].copy()
    
    # Remplacement des chaînes vides par NaN pour l'affichage Excel
    df_clean.replace("", np.nan, inplace=True) 

    final_df = df_clean
    
    # 5. Export
    output_file = os.path.join(path_folder_output, output_filename)
    os.makedirs(path_folder_output, exist_ok=True)
    final_df.to_excel(output_file, index=False)
    
    print(f"\nFichier Excel généré avec succès : {output_file}")




Ajustement du DataFrame : Conservation des lignes avec contenu pertinent...

Fichier Excel généré avec succès : C:/Users/FLORIAN/Desktop\Arborescences_Globales.xlsx


In [ ]:
# ----------------------------------------------------------------------
# 3. EXÉCUTION
# ----------------------------------------------------------------------

path_folder_output = "C:/Users/FLORIAN/Desktop"

# Configuration adaptée à ta structure :
# - __Autres : Profondeur 3 (Dossier -> Artiste -> Album)
# - Les autres : Profondeur 2 (Dossier -> Album)
folders_to_list = [
   ["M:/musiques/__Autres"    , 3, False],  
   ["M:/musiques/__B.O"       , 2, False],  
   ["M:/musiques/__CLASSIQUE" , 2, False],  
   ["M:/musiques/__COMPILS"   , 2, False],  
]

explorer_arborescences_global(folders_to_list, path_folder_output)

## Rechercher les 24 bit

In [ ]:
import os

# Définir le chemin de départ
base_path = "H:/Musique/__Autres/"

# Parcourir récursivement tous les dossiers
for root, dirs, files in os.walk(base_path):
    for dir_name in dirs:
        # Vérifier si "24bit" est dans le nom du dossier
        if "FLAC (" in dir_name.lower():  # .lower() pour rendre la recherche insensible à la casse
            # Afficher le chemin absolu du dossier
            print(os.path.abspath(os.path.join(root, dir_name)))
print("ok")

ok
